## 0. dependency
pip install datasets transformers accelerate peft trl bitsandbytes llmcompressor vllm matplotlib pandas numpy torch

## 1. data   
### sft dataset:  
for recovery after pruning   
### calibriation dataset:  
a small set of data to do forward to help the compressing algorithm see which parts of the model are okay to compress. mostly short data so it runs faster, but add some long data to improve distribution.

In [3]:
"""data - > data_process - > train_data - > calib_1000.jsonl & sft_10000.jsonl"""

'data - > data_process - > train_data - > calib_1000.jsonl & sft_10000.jsonl'

## 2. pruning
2.1. use llmcompressor to prune the model (keep 2 weights for every 4 weights).  
only pruning, without recovery. export as compressed model.    
2.2 then change it back to dense model so bitsandbytes can read it. 

In [4]:
"""pruning - > prune_qwen3_14b_2of4.py - > qwen3-14b-2of4-sparsegpt"""
"""pruning - > export_dense_from_pruned.py - > qwen3-14b-2of4-dense-bf16"""

'pruning - > export_dense_from_pruned.py - > qwen3-14b-2of4-dense-bf16'

## 3. recovery finetune
3.1. the performance of the model is hurt after pruning, need to finetune (with qlora) it to help it recover.  
first via instruction finetuning with the sft dataset.   
get lora adpater.  
3.2. merge the adapter with pruned model to get the final pruned model with instruction finetuning recovery.  

In [5]:
"""instruction_finetune - > sft_finetune.py - > qwen3-14b-pruned-sft-qlora_trainer"""
"""instruction_finetune - > merge_adapter_prune.py - > qwen3-14b-pruned-sft-merged-bf16"""

'instruction_finetune - > merge_adapter_prune.py - > qwen3-14b-pruned-sft-merged-bf16'

## 4. knowledge distillation
sentence level kd as an ablation to compare with instruction finetuning.   
not real knowledge distillation with KL divergence, but a teacher-forcing sft (because KL-KD is much more complicated and expensive).  
4.1. first generate kd dataset with teacher model (base model without pruning): use the same user questions as in the sft dataset, ask the base model to generate assistant answer    
4.2. do the same finetuning as in 3 (but essentially teacher forcing).   
4.3. merge lora adapter with pruned model

In [6]:
"""knowledge_distill - > make_kd_dataset - > kd_10000.jsonl"""
"""knowledge_distill - > kd_finetune.py - > qwen3-14b-pruned-kd-qlora_trainer"""
"""knowledge_distill - > merge_adapter_prune.py - > qwen3-14b-pruned-kd-merged-bf16"""

'knowledge_distill - > merge_adapter_prune.py - > qwen3-14b-pruned-kd-merged-bf16'

## 5. pre-quantization evaluation
collect evaluation dataset different from sft dataset.   
do eval to sft and kd recovery model respectively.  


In [7]:
"""pre_quant_eval - > eval.py"""

'pre_quant_eval - > eval.py'

## 6. quantization
do quant to both sft model and kd model.  
quantiztion with llmcompressor —— W4A16 (load the weights with 4-bits, but activate/infer with 16-bits to keep numerical stability)

In [8]:
"""quantization - > sft_quant.py - > qwen3-14b-pruned-sft-merged-AWQ-W4A16-G128"""
"""quantization - > kd_quant.py - > qwen3-14b-pruned-kd-merged-AWQ-W4A16-G128"""

'quantization - > kd_quant.py - > qwen3-14b-pruned-kd-merged-AWQ-W4A16-G128'

## 7. post-quantization evaluation
pruned model after sft and kd may act differently even with the same quantization methods.  
pre-quant performance is not deterministic for which one is better. 

In [ ]:
"""post_quant_eval - > eval.py"""